[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/15-projects/p09-support-inbox-automation/colab_quickstart.ipynb)

# P9 - Support-inbox automation: quick start
Triage ~200 synthetic support emails: extract fields into a Pydantic schema, find a help article, draft a reply, route (human approval / auto / spam), store in an SQLite approval queue, and write a daily summary. CPU runtime, no API key.

## 1. Get the code
In Colab this clones the course repo. If you are already inside the project folder (running locally), the clone is skipped.

In [1]:
import os, pathlib, subprocess, sys
PROJECT = "p09-support-inbox-automation"
if pathlib.Path("src").is_dir() and pathlib.Path("README.md").exists():
    print("Project folder found - using the local copy, no clone needed.")
else:
    if not pathlib.Path("AI-Engineer").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vvvvvivekkk/AI-Engineer.git"], check=True)
    os.chdir(f"AI-Engineer/15-projects/{PROJECT}")
print("working in", os.getcwd())

Project folder found - using the local copy, no clone needed.
working in /home/claude/AI-Engineer/15-projects/p09-support-inbox-automation


## 2. Install the pinned packages (Colab only)
Colab already has PyTorch with GPU support, so we skip the `torch` line to avoid replacing it. Locally you install with `pip install -r requirements.txt` in a venv.

In [2]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    reqs = [l for l in open("requirements.txt") if l.strip() and not l.startswith(("torch==", "#"))]
    open("/tmp/req.txt", "w").writelines(reqs)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/req.txt"], check=True)
print("ready")

ready


## 3. Generate emails and help articles
The emails include edge cases: two intents, missing order ids, angry or legal-threat tone, quoted old threads, spam, Hinglish.

In [3]:
!python -m src.make_emails
!python -m src.make_articles

wrote 200 emails -> data/emails.jsonl, 12 .eml files -> data/inbox/
edge cases: {'quoted_thread': 21, 'two_intents': 15, 'phone_in_signature': 27, 'missing_order_id': 82, 'spam': 16, 'angry': 26, 'safety': 5, 'legal_threat': 11, 'very_short': 13, 'hinglish': 5}


wrote 15 articles -> data/help_articles/


## 4. Run the Python pipeline on all emails
Prints the route counts and accuracy against the true labels.

In [4]:
!python -m src.pipeline --fresh

processed 200 emails | routes {'human_review': 109, 'discard': 16, 'auto': 75} | extract methods {'rules': 200}
accuracy vs true labels: {'intents_exact': 0.99, 'urgency': 1.0, 'order_id': 1.0, 'product': 1.0, 'route': 1.0}

example -> route=human_review reasons=['money involved']
{
 "customer_name": "Uma Das",
 "customer_email": "uma.das@gmail.com",
 "order_id": "ORD-41820",
 "product": "SmartKettle 2",
 "intents": [
  "billing_issue"
 ],
 "urgency": "medium",
 "sentiment": "negative",
 "summary": "billing issue about SmartKettle 2 (order ORD-41820)"
}
Hi Uma,

Thanks for contacting GadgetCo. We're really sorry about this experience. I've found your order ORD-41820. A second charge is often a temporary card authorisation that disappears within 3 working days. If it is still there after 3 days, reply with your bank statement line and we refund it. 

Best wishes,
GadgetCo Support


## 5. Watch an inbox folder
The same pipeline reading `.eml` files (already-processed emails are skipped thanks to the unique email id).

In [5]:
!python -m src.pipeline --inbox data/inbox

processed 12 emails | routes {'human_review': 9, 'auto': 2, 'discard': 1} | extract methods {'rules': 12}

example -> route=human_review reasons=['high urgency', 'angry customer']
{
 "customer_name": "Tom Baker",
 "customer_email": "tom.baker@gmail.com",
 "order_id": null,
 "product": "FitBand Pro",
 "intents": [
  "technical_support"
 ],
 "urgency": "high",
 "sentiment": "angry",
 "summary": "technical support about FitBand Pro (no order id)"
}
Hi Tom,

Thanks for contacting GadgetCo. We're really sorry about this experience. Could you reply with your order number (it looks like ORD-12345)? Turn on Bluetooth, open the app and choose Add device. If pairing fails, restart the band by holding the button for 10 seconds and try again. 

Best wishes,
GadgetCo Support


## 6. Be the human reviewer
List pending items through the API and approve one.

In [6]:
from fastapi.testclient import TestClient
from src.api import app
api = TestClient(app)
pending = api.get("/queue").json()["pending"]
print(len(pending), "tickets waiting. First one:")
first = pending[0]
print(first["subject"], "|", first["reasons"]) ; print(first["draft"])
print(api.post(f"/queue/{first['id']}/approve", json={"reviewer": "trainer", "note": "looks good"}).json()["status"])

109 tickets waiting. First one:
Billing problem | ["money involved"]
Hi Uma,

Thanks for contacting GadgetCo. We're really sorry about this experience. I've found your order ORD-41820. A second charge is often a temporary card authorisation that disappears within 3 working days. If it is still there after 3 days, reply with your bank statement line and we refund it. 

Best wishes,
GadgetCo Support
approved


## 7. Daily summary report

In [7]:
!python -m src.report --date 2026-11-20

# Support inbox - daily summary for 2026-11-20

- Emails received: **39**
- Waiting for human approval (this day): **19** (all days: 108)
- Auto-filed: **16** · Spam discarded: **4** · Approved: **0** · Rejected: **0**

## What customers wanted
| intent | count |
|---|---|
| delivery_status | 6 |
| refund | 5 |
| technical_support | 5 |
| damaged_item | 5 |
| feedback | 4 |
| spam | 4 |
| account_access | 4 |
| cancel_order | 4 |
| billing_issue | 3 |

## Urgency
| urgency | count |
|---|---|
| medium | 14 |
| low | 13 |
| high | 12 |

## Products mentioned
| product | count |
|---|---|
| SoundBuds X | 8 |
| AquaPure Filter | 8 |
| FitBand Pro | 5 |
| HomeCam 360 | 5 |
| SmartKettle 2 | 4 |

## Why emails were sent to a human
| reason | count |
|---|---|
| high urgency | 12 |
| money involved | 8 |
| angry customer | 8 |
| order id missing | 5 |
| spam | 4 |
| possible safety issue | 2 |
| legal / chargeback threat | 1 |
| multiple intents | 1 |

## Top urgent items waiting for approva

## 8. The n8n workflow
The importable file is `n8n/support_triage_workflow.json`. n8n cannot run in Colab; on your own computer run n8n with Docker and import it (see README).

In [8]:
import json
wf = json.load(open("n8n/support_triage_workflow.json"))
for n in wf["nodes"]:
    print(f"{n['name']:28s} {n['type']}")

IMAP Email Trigger           n8n-nodes-base.emailReadImap
Manual Trigger (test)        n8n-nodes-base.manualTrigger
Sample email                 n8n-nodes-base.set
Map email fields             n8n-nodes-base.set
POST /triage                 n8n-nodes-base.httpRequest
Needs human approval?        n8n-nodes-base.if
Alert reviewers (Slack)      n8n-nodes-base.slack
Auto-filed (no action)       n8n-nodes-base.noOp
Every day at 18:00           n8n-nodes-base.scheduleTrigger
GET /report/daily            n8n-nodes-base.httpRequest
Email report to manager      n8n-nodes-base.emailSend


## 9. Run the tests

In [9]:
!python -m pytest -q tests

......

..                                                                 [100%]
8 passed in 0.41s
